In [36]:
import pandas as pd
import json

# ============================
# 1. LOAD USDA NUTRITION DATA
# ============================

def load_usda_nutrition(json_path):
    """Load USDA Foundation Foods JSON and extract kcal + protein per 100g."""
    with open(json_path, "r") as f:
        data = json.load(f)

    # Handle both list and wrapped formats
    if isinstance(data, dict) and "FoundationFoods" in data:
        foods = data["FoundationFoods"]
    elif isinstance(data, list):
        foods = data
    else:
        raise ValueError("Unrecognized USDA JSON structure")

    rows = []
    for food in foods:
        desc = food.get("description")
        fdc_id = food.get("fdcId")

        kcal = None
        protein = None

        for fn in food.get("foodNutrients", []):
            name = (fn.get("nutrient", {}).get("name") or "").lower()
            unit = (fn.get("nutrient", {}).get("unitName") or "").lower()
            amount = fn.get("amount")

            if amount is None:
                continue

            if "energy" in name and unit == "kcal":
                kcal = amount
            if "protein" in name and unit == "g":
                protein = amount

        if kcal is not None and protein is not None:
            rows.append({
                "fdc_id": fdc_id,
                "food_name": desc,
                "calories_100g": kcal,
                "protein_100g": protein
            })

    return pd.DataFrame(rows)

# Example:
# usda_df = load_usda_nutrition("FoodData_Central_foundation_food_json_2025-12-18.json")
# ============================
# 2. LOAD FAH CATEGORY DICTIONARY (Sheet 1)
# ============================

# Load metadata sheet (Sheet 1)
efpg_dict = pd.read_csv("FMAP.csv", skiprows=23,skipfooter=17)
# efpg_dict.columns
# Clean up column names
efpg_dict.columns = efpg_dict.columns.str.strip()

# Keep only the category dictionary rows
efpg_dict = efpg_dict[["EFPG_code", "EFPG_name", "Tier 1 group", "Tier 2 group"]]

# Convert EFPG_code to int
efpg_dict["EFPG_code"] = efpg_dict["EFPG_code"].astype(int)

# efpg_dict.head()

# ============================
# 3. LOAD FAH MONTHLY PRICE DATA (Sheet 2)
# ============================

# This is the large sheet that was too big to preview earlier
fmap_data = pd.read_csv("FMAP_data.csv")

# Keep only the columns we need
fmap_data = fmap_data[[
    "Year", "Month", "EFPG_code", "Metroregion_code",
    "Unit_value_mean_wtd", "Purchase_grams_wtd"
]]

# Price per 100g is already provided by USDA
fmap_data.rename(columns={"Unit_value_mean_wtd": "price_per_100g_usd"}, inplace=True)

# fmap_data.head()
# Merge price data with category names
fmap = fmap_data.merge(efpg_dict, on="EFPG_code", how="left")

# fmap.head()

def simple_similarity(a, b):
    a = a.lower()
    b = b.lower()

    # Token overlap
    tokens_a = set(a.split())
    tokens_b = set(b.split())
    overlap = len(tokens_a & tokens_b)

    # Longest common substring
    longest = 0
    for i in range(len(a)):
        for j in range(i+1, len(a)+1):
            if a[i:j] in b and len(a[i:j]) > longest:
                longest = len(a[i:j])

    return overlap * 20 + longest

PRIORITY_KEYWORDS = {
    # Cheese before milk
    "ricotta": "Cheese and cream cheese",
    "cheese": "Cheese and cream cheese",
    "mozzarella": "Cheese and cream cheese",
    "cheddar": "Cheese and cream cheese",

    # Yogurt / cream
    "yogurt": "Whole yogurt",
    "sour cream": "Whole cream and sour cream",

    # Milk (lower priority)
    "whole milk": "Whole milk",
    "skim milk": "Reduced-fat, low-fat, and skim milk",
    "milk": "Whole milk",

    # Meat
    "chicken breast": "Chicken, turkey, and game birds, fresh",
    "chicken": "Chicken, turkey, and game birds, fresh",

    # Fruit
    "apple": "Whole fruit, fresh",

    # Grains
    "rice": "Non-whole-grain rice and pasta",
}

def fuzzy_fallback(food_name, fah_categories, threshold=25):
    best_cat = None
    best_score = 0

    for cat in fah_categories:
        score = simple_similarity(food_name, cat)
        if score > best_score:
            best_score = score
            best_cat = cat

    return best_cat if best_score >= threshold else None

def map_food_to_fah(food_name, fah_categories, priority_map=PRIORITY_KEYWORDS):
    food_l = food_name.lower()

    # Priority rules
    for keyword, category in priority_map.items():
        if keyword in food_l:
            return category, "priority"

    # Fuzzy fallback
    fuzzy = fuzzy_fallback(food_name, fah_categories)
    if fuzzy:
        return fuzzy, "fuzzy"

    return None, "none"
def auto_map_usda_to_fah(usda_df, efpg_dict):
    fah_categories = efpg_dict["EFPG_name"].dropna().unique().tolist()

    results = []
    for _, row in usda_df.iterrows():
        food = row["food_name"]
        cat, method = map_food_to_fah(food, fah_categories)
        results.append({
            "food_name": food,
            "fah_category": cat,
            "mapping_method": method
        })

    return pd.DataFrame(results)
def map_food_to_fah_category(food_name, category_map):
    food_l = food_name.lower()
    for key, val in category_map.items():
        if key in food_l:
            return val
    return None

def compute_nutrition_vs_cost(usda_row, fah_df):
    fah_cat = usda_row["fah_category"]
    if pd.isna(fah_cat):
        return None

    # national average price
    subset = fah_df[(fah_df["EFPG_name"] == fah_cat) & (fah_df["Metroregion_code"] == 0)]
    if subset.empty:
        return None

    price = subset["price_per_100g_usd"].mean()

    return {
        "food_name": usda_row["food_name"],
        "fah_category": fah_cat,
        "avg_price_per_100g_usd": price,
        "calories_per_dollar": usda_row["calories_100g"] / price,
        "protein_g_per_dollar": usda_row["protein_100g"] / price
    }
def print_result_pretty(result):
    print(f"Food:                 {result['food_name']}")
    print(f"FAH Category:         {result['fah_category']}")
    print(f"Avg Price / 100g:     ${result['avg_price_per_100g_usd']:.3f}")
    print(f"Calories per $:       {result['calories_per_dollar']:.1f}")
    print(f"Protein (g) per $:    {result['protein_g_per_dollar']:.1f}")

/tmp/ipykernel_25000/4079988086.py:59: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support skipfooter; you can avoid this warning by specifying engine='python'.
  efpg_dict = pd.read_csv("FMAP.csv", skiprows=23,skipfooter=17)


In [37]:
usda_df = load_usda_nutrition("FoodData_Central_foundation_food_json_2025-12-18.json")
# ============================
# RUN A SEARCH
# ============================

search_term = "cheese"   # <-- PUT YOUR SEARCH TERM HERE

# 1. Find USDA food
usda_match = usda_df[usda_df["food_name"].str.contains(search_term, case=False)]

if usda_match.empty:
    print("No USDA match found")
else:
    usda_row = usda_match.iloc[0]

    # 2. Map to FAH category
    fah_cat, method = map_food_to_fah(
        usda_row["food_name"],
        efpg_dict["EFPG_name"].tolist()
    )

    print("Mapped to:", fah_cat, "(method:", method, ")")

    # 3. Compute nutrition vs cost
    result = compute_nutrition_vs_cost(
        {
            "food_name": usda_row["food_name"],
            "fah_category": fah_cat,
            "calories_100g": usda_row["calories_100g"],
            "protein_100g": usda_row["protein_100g"]
        },
        fmap
    )

    print_result_pretty(result)

Mapped to: Cheese and cream cheese (method: priority )
Food:                 Cheese, parmesan, grated
FAH Category:         Cheese and cream cheese
Avg Price / 100g:     $1.102
Calories per $:       381.9
Protein (g) per $:    26.9
